# Create VAE for the different words sin disntace functions

In [ ]:
# Add import 
import sys
import torch 
from torch import nn
from torch import optim
from prodigyopt import Prodigy # proddigy optimizer from https://github.com/konstmish/prodigy?tab=readme-ov-file
import lightning.pytorch as pl

import tqdm
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# allow reload of python modules
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
from dataset.RobotPathDataset.normalizer import MinMaxFeatureNormalizer

from model.models import EMA
import copy
import time

# remove all warnings
import warnings
warnings.filterwarnings("ignore")

In [32]:
normalizer = MinMaxFeatureNormalizer()
encoder_normalizer = MinMaxFeatureNormalizer()
representation_dim = 128
CONFIG = {
    # Model configuration
    'representation_dim': representation_dim,
    'dim_mults':(1,4,8), # Dimension multipliers for hidden layers of the model # (1, 4, 8)
    
    # Encoder configuration
    'embeder_num_of_hidden_layers' : 1,
    'vae_kwargs': {
        'in_channels':1,
        'latent_dim': representation_dim,
        'hidden_dims': [8, 16, 32, 64, 128],
    },
    
    # Hyperparameters for training
    'batch_size': 32,
    'num_epochs': 2000,
    'ema': EMA(beta=0.99), # Exponential moving average for the model weights
        ## Optimizer configuration
        'optimizer': Prodigy,
        'optimizer_kwargs': {
            'lr': 1., # ! ONLY FOR PRODIGY OPTIMIZER
            'weight_decay': 0.01, 
            'safeguard_warmup':True,
            'use_bias_correction':True,
            'betas': (0.9, 0.99),
            },
        # Scheduler configuration
        'scheduler': torch.optim.lr_scheduler.CosineAnnealingLR,
        'scheduler_kwargs': {
            # 'gamma': 0.999,
            'T_max': 10, # Total number of iterations
        },
    
    # Hyperparameters for diffusion process
    'noise_steps': 256,
    'normalize': True,
    'normalizer':normalizer,
    'encoder_normalizer':encoder_normalizer,
    'cfg_scale': 3,


    # Dataset specific configuration
    'n_paths_per_world': 10,
    'n_worlds': 10,
    'n_waypoints': 64, # due to the archtechture has to be a number that is a power of 2 
}

# DataLoader

### Use the other dataset class to ready the data then get the worlds images from it

In [33]:
from dataset import RobotPathDataset
file = 'data/SingleSphere02_all.db'
# file = 'D:\Desktop\ADLR\RobotPathData\data\SingleSphere02_one-world.db'
dataset = RobotPathDataset(file, n_paths_per_world=CONFIG['n_paths_per_world'], n_worlds=CONFIG['n_worlds'],  n_waypoints=CONFIG['n_waypoints'],   normalizer=CONFIG['normalizer'])
print(f' sample shape {dataset[0]["path"].shape} with {len(dataset)} samples')

data shape: torch.Size([100, 64, 2]), min_values: tensor([0, 0], device='cuda:0'), max_values: tensor([10, 10], device='cuda:0'), n_worlds:8, samples: 100
 sample shape torch.Size([64, 2]) with 100 samples


In [34]:
from dataset.RobotPathDataset.obstacle_distance import img2dist_img

In [35]:
class WorldsDataset(torch.utils.data.Dataset):
    def __init__(self, dataset:RobotPathDataset, sign_dist_field=False):
        worlds_dataset = dataset.all_world_images
        self.data = []
        
        self.voxel_size = 10 / 64     # in m

        if sign_dist_field:
            for i in range(len(worlds_dataset)):
                dist_field= torch.tensor(img2dist_img(img=worlds_dataset[i], voxel_size=self.voxel_size, add_boundary=True), dtype=torch.float32)
                self.data.append(dist_field)
                
        self.data = torch.stack(self.data)
        self.data = self.data.unsqueeze(1)
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

In [36]:
world_dataset = WorldsDataset(dataset, sign_dist_field=True)
print(f' sample shape {world_dataset[0].shape} with {len(world_dataset)} samples')

 sample shape torch.Size([1, 64, 64]) with 10000 samples


In [37]:
sample = world_dataset[9]
sample

tensor([[[0.1562, 0.1562, 0.1562,  ..., 0.1562, 0.1562, 0.1562],
         [0.1562, 0.3125, 0.3125,  ..., 0.3125, 0.3125, 0.1562],
         [0.1562, 0.3125, 0.4688,  ..., 0.4688, 0.3125, 0.1562],
         ...,
         [0.1562, 0.3125, 0.4688,  ..., 0.1562, 0.3125, 0.1562],
         [0.1562, 0.3125, 0.3125,  ..., 0.1562, 0.3125, 0.1562],
         [0.1562, 0.1562, 0.1562,  ..., 0.1562, 0.1562, 0.1562]]])

In [38]:
plt.imshow(sample.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')

### Dataloader

In [39]:
from torch.utils.data.dataset import random_split
train_dataset, val_dataset = random_split(world_dataset, [0.9, 0.1])

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=True)
sample_batch = next(iter(train_dataloader))

# Model

In [40]:
from model.vae import VanillaVAE, BaseVAE

In [41]:
normalizer = MinMaxFeatureNormalizer()
model = VanillaVAE(**CONFIG['vae_kwargs'], normalizer=normalizer)
model

VanillaVAE(
  (encoder): Sequential(
    (0): Sequential(
      (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
    )
    (1): Sequential(
      (0): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
    )
    (2): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
    )
    (3): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01)
    )
    (4): Sequential

In [42]:
normalized_sample_batch = normalizer.normalize(sample_batch)

In [43]:
mu, log_var = model.encode(sample_batch)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample_batch.shape , f'{reconstructed.shape} != {sample_batch.shape}'

In [44]:
results = model.forward(sample_batch)
results
model.loss_function(*results, M_N=1)

{'loss': tensor(18.5428, grad_fn=<AddBackward0>),
 'Reconstruction_Loss': tensor(2.4896),
 'KLD': tensor(-16.0532)}

In [45]:
# visualize const vs reconstructed
fig, ax = plt.subplots(1,3, figsize=(10,5))
pos = ax[0].imshow(sample_batch[0].squeeze().T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])
pos = ax[2].imshow(normalized_sample_batch[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[2].set_title('Normalized')
fig.colorbar(pos, ax=ax[2])

In [46]:
from model.vae import VanillaVAE, BaseVAE, VAEXperiment
autoencoder = VAEXperiment(model, CONFIG['vae_kwargs'])

In [47]:
# DEBUG lightning before running full training
trainer = pl.Trainer(limit_train_batches=64, max_epochs=2)
trainer.fit(model=autoencoder, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | VanillaVAE | 395 K  | train
---------------------------------------------
395 K     Trainable params
0         Non-trainable params
395 K     Total params
1.580     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


In [48]:
import os
from lightning.pytorch import loggers
logger = loggers.TensorBoardLogger(os.getcwd(), name='vae')
trainer = pl.Trainer(enable_progress_bar=False, enable_checkpointing=True, logger=logger, max_epochs=CONFIG['num_epochs'])
trainer.fit(model=autoencoder, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | VanillaVAE | 395 K  | train
---------------------------------------------
395 K     Trainable params
0         Non-trainable params
395 K     Total params
1.580     Total estimated model params size (MB)


In [ ]:
model = autoencoder.model
model.eval()
sample = world_dataset[9]
sample = sample.unsqueeze(0)
# sample = sample.to(autoencoder.curr_device)
mu, log_var = model.encode(sample)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample.shape , f'{reconstructed.shape} != {sample.shape}'
# visualize const vs reconstructed
fig, ax = plt.subplots(1,2, figsize=(10,5))
pos = ax[0].imshow(sample[0].squeeze().T.cpu().detach(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])


In [ ]:
# reload model form checkpoint
model_old = VAEXperiment.load_from_checkpoint(r'/home/karim.samir.lotfy/tum-adlr-ss24-09/lightning_logs/version_10/checkpoints/epoch=227-step=256500.ckpt')

In [ ]:
model_old.freeze()
vae_model = model_old.model


In [ ]:
vae_model.eval()
sample = world_dataset[88]
sample = sample.unsqueeze(0)
sample = sample.to(autoencoder.curr_device)
mu, log_var = model.encode(sample)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample.shape , f'{reconstructed.shape} != {sample.shape}'
# visualize const vs reconstructed
fig, ax = plt.subplots(1,2, figsize=(10,5))
pos = ax[0].imshow(sample[0].squeeze().T.cpu().detach(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])

In [ ]:
plt.show()